# 05. Workplace Safety and Discrimination Harmonization (2014–2024)

This notebook processes historical survey datasets from the Finnish Social Science Data Archive (FSD). It extracts, cleans, and harmonizes key indicators regarding workplace safety and equality across survey years (2014–2024):

1. **Discrimination** (Reported experiences based on survey items such as K33, K33A, and K67)
2. **Bullying** (Mental violence and psychological harassment by colleagues or management, items from K9 series)
3. **Physical Violence** (Threats and physical incidents at the workplace, items from VAK / 10B series)

In [1]:
# ==========================================
# 1. IMPORTS & CONFIGURATION (2014–2024)
# ==========================================

import glob
import os
import pandas as pd
import numpy as np

# Simple dictionary of survey years (using clean FSD numbers without 'daF')
fsd_year_mapping = {
    '3049': 2014,
    '3131': 2015,
    '3220': 2016,
    '3481': 2017,
    '3482': 2018,
    '3493': 2019,
    '3645': 2020,
    '3727': 2021,
    '3784': 2022,
    '3914': 2023,
    '4011': 2024,
}

In [2]:
# ==========================================
# 2. HELPER FUNCTIONS FOR PROCESSING
# ==========================================

# Function to find year from filename
def get_year(filename):
    for code, year in fsd_year_mapping.items():
        if code in filename:
            return year
    return None

# Function to load CSV with multi-separator fallback
def load_raw_csv(filepath):
    for sep in [';', ',']:
        try:
            df = pd.read_csv(filepath, sep=sep, low_memory=False, encoding='utf-8-sig')
            if df.shape[1] > 1:
                return df
        except Exception:
            continue
    return pd.read_csv(filepath, low_memory=False)

print("Block 2 executed: Helper functions ready.")

Block 2 executed: Helper functions ready.


In [3]:
# ==========================================
# 3. DATA HARMONIZATION & PROCESSING (2014–2024)
# ==========================================

# 1. Locate all CSV survey files in the data directory
data_dir = "data"
all_files = glob.glob(os.path.join(data_dir, "daF*.csv"))

all_years_data = []

# 2. Process each file sequentially
for file in sorted(all_files):
    year = get_year(file) 
    if not year:
        continue
        
    df = load_raw_csv(file)
    if df is None:
        continue
        
    # Standardize column names to uppercase
    df.columns = [str(c).upper() for c in df.columns]
    
    temp_df = pd.DataFrame()
    temp_df["Year"] = [year] * len(df)
    
    # --- 1. Discrimination (Universal search across all years) ---
    if year == 2021:
        disc_cols = [c for c in df.columns if c == "K67"]
    elif year >= 2022:
        disc_cols = [c for c in df.columns if c == "K33A"]
    else:
        disc_cols = [c for c in df.columns if c == "K33" or c.startswith("K33_")]

    if disc_cols:
        sub_dfs = []
        for c in disc_cols:
            s = pd.to_numeric(df[c], errors="coerce")
            sub_dfs.append(s)
        combined_disc = pd.concat(sub_dfs, axis=1)
        # Recode explicitly to float: 1 = Yes -> 1.0, 2 = No -> 0.0
        temp_df["Discrimination"] = combined_disc.apply(
            lambda row: 1.0 if (row == 1).any() else (0.0 if (row == 2).any() else np.nan), axis=1
        )
    else:
        temp_df["Discrimination"] = np.nan

    # --- 2. Bullying / Mental Violence ---
    bully_col = next((c for c in df.columns if "K9" in c and ("07" in c or "_7" in c)), None)
    if bully_col:
        s = pd.to_numeric(df[bully_col], errors="coerce")
        # Recode: 1 = No -> 0.0, 2 or 3 = Yes -> 1.0
        temp_df["Bullying"] = s.map({1.0: 0.0, 2.0: 1.0, 3.0: 1.0})
    else:
        temp_df["Bullying"] = np.nan

    # --- 3. Physical Violence ---
    phys_col = next((c for c in df.columns if "VAK" in c or "10B" in c), None)
    if phys_col:
        s = pd.to_numeric(df[phys_col], errors="coerce")
        temp_df["Physical_Violence"] = s.map({1.0: 0.0, 2.0: 1.0, 3.0: 1.0})
    else:
        temp_df["Physical_Violence"] = np.nan

    all_years_data.append(temp_df)

# 3. Concatenate all yearly slices into a single master dataframe
master_df = pd.concat(all_years_data, ignore_index=True)

print("\n--- Summary Table: Share of 'Yes' Responses (%) ---")
# Calculate percentage of 'Yes' responses grouped by year
summary_table = master_df.groupby("Year", dropna=False).mean(numeric_only=True) * 100
print(summary_table.to_string())


--- Summary Table: Share of 'Yes' Responses (%) ---
      Discrimination   Bullying  Physical_Violence
Year                                              
2014       26.763848  39.196273          10.058140
2015       31.123919  37.744534           8.568143
2016       29.975430  37.084871           8.184615
2017       28.486998  38.388626           8.698225
2018       29.550971  37.378641          10.753341
2019       28.185328  37.990985          10.587476
2020       27.399757  35.579842           9.063260
2021       14.727794  35.074217           6.965699
2022        8.619613  37.442661           7.704741
2023        7.809748  35.612083           6.112224
2024        9.805014  36.613154           7.793473


In [4]:
# ==========================================
# 4. EXPORT HARMONIZED DATA
# ==========================================

output_filename = "harmonized_safety_indicators.csv"
master_df.to_csv(output_filename, index=False)

print(f"Master dataset successfully saved to project root: {output_filename}")

Master dataset successfully saved to project root: harmonized_safety_indicators.csv
